# Imports

In [225]:
from feature_engineering.config import PROCESSED_DATA_DIR, INTERIM_DATA_DIR

import os
import pandas
import numpy as np

# Ingest data

In [101]:
os.listdir(INTERIM_DATA_DIR)

['.gitkeep', 'data_manifest.json', 'meses.parquet', 'nomenclatura.parquet']

In [140]:
filepath_months = INTERIM_DATA_DIR / 'meses.parquet'
filepath_nom = INTERIM_DATA_DIR / 'nomenclatura.parquet'

df_months = pandas.read_parquet(filepath_months)
df_nom = pandas.read_parquet(filepath_nom)

In [103]:
print(df_months.shape)
df_months.head()

(8943941, 8)


c:\Users\gsaen\Documents\Maestria ITESM\4. Trimestre 4 (Sep-Nov 2026)\Materias\MLOps\Proyecto\mlops\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,viaje_id,usuario_id,genero,ano_de_nacimiento,inicio_del_viaje,fin_del_viaje,origen_id,destino_id
0,35406564,3366663,M,2000.0,01/09/2024 0:00,01/09/2024 0:07,154,161
1,35406567,631455,M,1990.0,01/09/2024 0:02,01/09/2024 0:08,50,156
2,35406571,216004,M,1990.0,01/09/2024 0:02,01/09/2024 0:18,64,273
3,35406572,676145,M,1988.0,01/09/2024 0:03,01/09/2024 0:10,271,164
4,35406573,2244256,M,1998.0,01/09/2024 0:03,01/09/2024 0:19,199,63


In [104]:
df_months.info()

<class 'pandas.DataFrame'>
RangeIndex: 8943941 entries, 0 to 8943940
Data columns (total 8 columns):
 #   Column             Dtype  
---  ------             -----  
 0   viaje_id           int64  
 1   usuario_id         int64  
 2   genero             str    
 3   ano_de_nacimiento  float16
 4   inicio_del_viaje   str    
 5   fin_del_viaje      str    
 6   origen_id          int64  
 7   destino_id         int64  
dtypes: float16(1), int64(4), str(3)
memory usage: 823.4 MB


In [105]:
df_months.describe().T

c:\Users\gsaen\Documents\Maestria ITESM\4. Trimestre 4 (Sep-Nov 2026)\Materias\MLOps\Proyecto\mlops\Lib\site-packages\pandas\core\nanops.py:1524: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
c:\Users\gsaen\Documents\Maestria ITESM\4. Trimestre 4 (Sep-Nov 2026)\Materias\MLOps\Proyecto\mlops\Lib\site-packages\numpy\_core\_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
c:\Users\gsaen\Documents\Maestria ITESM\4. Trimestre 4 (Sep-Nov 2026)\Materias\MLOps\Proyecto\mlops\Lib\site-packages\pandas\core\nanops.py:725: RuntimeWarning: invalid value encountered in scalar divide
  the_mean = the_sum / count if count > 0 else np.nan


,count,mean,std,min,25%,50%,75%,max
viaje_id,8943941.0,4.053086e+07,2.970618e+06,35406564.0,37963087.0,40530663.0,43073755.0,45705292.0
usuario_id,8943941.0,2.473625e+06,1.726834e+06,102.0,636644.0,2414404.0,4467406.0,4770514.0
ano_de_nacimiento,8941893.0,NaN,0.000000e+00,199.0,1985.0,1993.0,1999.0,2023.0
origen_id,8943941.0,1.574391e+02,1.153549e+02,2.0,51.0,157.0,258.0,519.0
destino_id,8943941.0,1.593281e+02,1.182521e+02,2.0,51.0,157.0,263.0,519.0


In [106]:
print(df_nom.shape)
df_nom.head()

(484, 5)


,id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


In [107]:
df_nom.info()

<class 'pandas.DataFrame'>
RangeIndex: 484 entries, 0 to 483
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          484 non-null    int64  
 1   name        484 non-null    str    
 2   latitude    484 non-null    float64
 3   longitude   484 non-null    float64
 4   dpcapacity  484 non-null    int64  
dtypes: float64(2), int64(2), str(1)
memory usage: 38.9 KB


In [108]:
df_nom.id

0        2
1        3
2        4
3        5
4        6
      ... 
479    515
480    516
481    517
482    518
483    519
Name: id, Length: 484, dtype: int64

# Remove unused IDs

In [141]:
nom_id_list = df_nom.index.to_list()
df_filtered = df_months.query(
    '(origen_id in @nom_id_list) & (destino_id in @nom_id_list)'
)

print('Original row count:', df_months.shape[0])
print('Filtered row count:', df_filtered.shape[0])
print('Diff: ', df_months.shape[0] - df_filtered.shape[0])

Original row count: 8943941
Filtered row count: 8926227
Diff:  17714


# New features

## P1 - Regression

¿Cuántos viajes se iniciarán en una estación o zona en un periodo futuro? 

Variable objetivo: Suma de los viajes iniciados en cada estación durante cada hora.

Para construirla, la hora de inicio de cada viaje se redondea hacia abajo a la hora completa (por ejemplo, 08:47 pasa a 08:00), y se agrupan los viajes por estación de origen y hora. Las combinaciones de estación de origen y hora que no posean viaje, se registran como cero.

### Convert inicio_del_viaje to datetime

In [143]:
df_p1 = df_filtered.copy()

In [142]:
df_idv = df_filtered[['inicio_del_viaje']].copy()
df_idv['len_str'] = df_idv.inicio_del_viaje.str.len()

df_idv.len_str.unique()

array([15, 16, 19])

In [144]:
for i in [15, 16, 19]:
    display( df_idv
        .query('len_str == @i')
        .iloc[:2]
        .reset_index(drop=True)
    )

,inicio_del_viaje,len_str
0,01/09/2024 0:00,15
1,01/09/2024 0:02,15


,inicio_del_viaje,len_str
0,01/09/2024 10:00,16
1,01/09/2024 10:00,16


,inicio_del_viaje,len_str
0,2024-11-01 00:00:04,19
1,2024-11-01 00:00:14,19


In [145]:
dates_f1_series = pandas.to_datetime(
    df_idv.query('inicio_del_viaje.str.len() < 19').inicio_del_viaje,
    format='%d/%m/%Y %H:%M'
)

dates_f1_series

0        2024-09-01 00:00:00
1        2024-09-01 00:02:00
2        2024-09-01 00:02:00
3        2024-09-01 00:03:00
4        2024-09-01 00:03:00
                 ...        
808757   2024-10-31 23:58:00
808758   2024-10-31 23:58:00
808759   2024-10-31 23:59:00
808760   2024-10-31 23:59:00
808761   2024-10-31 23:59:00
Name: inicio_del_viaje, Length: 808762, dtype: datetime64[us]

In [146]:
dates_f2_series = pandas.to_datetime(
    df_idv.query('inicio_del_viaje.str.len() == 19').inicio_del_viaje,
    format='%Y-%m-%d %H:%M:%S'
)

dates_f2_series

808762    2024-11-01 00:00:04
808763    2024-11-01 00:00:14
808764    2024-11-01 00:00:16
808765    2024-11-01 00:00:26
808766    2024-11-01 00:01:54
                  ...        
8943936   2026-08-31 23:58:29
8943937   2026-08-31 23:58:53
8943938   2026-08-31 23:58:56
8943939   2026-08-31 23:59:27
8943940   2026-08-31 23:59:36
Name: inicio_del_viaje, Length: 8117465, dtype: datetime64[us]

In [147]:
dates_f1_series.name = 'date_f1'
dates_f2_series.name = 'date_f2'

In [148]:
df_idv = df_idv.join([dates_f1_series, dates_f2_series])
df_idv

,inicio_del_viaje,len_str,date_f1,date_f2
0,01/09/2024 0:00,15,2024-09-01 00:00:00,NaT
1,01/09/2024 0:02,15,2024-09-01 00:02:00,NaT
2,01/09/2024 0:02,15,2024-09-01 00:02:00,NaT
3,01/09/2024 0:03,15,2024-09-01 00:03:00,NaT
4,01/09/2024 0:03,15,2024-09-01 00:03:00,NaT
...,...,...,...,...
8943936,2026-08-31 23:58:29,19,NaT,2026-08-31 23:58:29
8943937,2026-08-31 23:58:53,19,NaT,2026-08-31 23:58:53
8943938,2026-08-31 23:58:56,19,NaT,2026-08-31 23:58:56
8943939,2026-08-31 23:59:27,19,NaT,2026-08-31 23:59:27


In [149]:
df_p1['inicio_del_viaje'] = df_idv.date_f1.fillna( df_idv.date_f2 )
df_p1

c:\Users\gsaen\Documents\Maestria ITESM\4. Trimestre 4 (Sep-Nov 2026)\Materias\MLOps\Proyecto\mlops\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,viaje_id,usuario_id,genero,ano_de_nacimiento,inicio_del_viaje,fin_del_viaje,origen_id,destino_id
0,35406564,3366663,M,2000.0,2024-09-01 00:00:00,01/09/2024 0:07,154,161
1,35406567,631455,M,1990.0,2024-09-01 00:02:00,01/09/2024 0:08,50,156
2,35406571,216004,M,1990.0,2024-09-01 00:02:00,01/09/2024 0:18,64,273
3,35406572,676145,M,1988.0,2024-09-01 00:03:00,01/09/2024 0:10,271,164
4,35406573,2244256,M,1998.0,2024-09-01 00:03:00,01/09/2024 0:19,199,63
...,...,...,...,...,...,...,...,...
8943936,45705286,2211176,M,1995.0,2026-08-31 23:58:29,2026-09-01 00:09:52,408,72
8943937,45705287,4612440,M,1990.0,2026-08-31 23:58:53,2026-09-01 00:05:20,32,74
8943938,45705288,4745711,M,1989.0,2026-08-31 23:58:56,2026-08-31 23:59:04,386,386
8943939,45705291,4745711,M,1989.0,2026-08-31 23:59:27,2026-09-01 00:08:31,386,384


### Group by hour and station

In [150]:
df_p1['hora_del_viaje'] = df_p1.inicio_del_viaje.dt.floor('h')

df_p1

c:\Users\gsaen\Documents\Maestria ITESM\4. Trimestre 4 (Sep-Nov 2026)\Materias\MLOps\Proyecto\mlops\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,viaje_id,usuario_id,genero,ano_de_nacimiento,inicio_del_viaje,fin_del_viaje,origen_id,destino_id,hora_del_viaje
0,35406564,3366663,M,2000.0,2024-09-01 00:00:00,01/09/2024 0:07,154,161,2024-09-01 00:00:00
1,35406567,631455,M,1990.0,2024-09-01 00:02:00,01/09/2024 0:08,50,156,2024-09-01 00:00:00
2,35406571,216004,M,1990.0,2024-09-01 00:02:00,01/09/2024 0:18,64,273,2024-09-01 00:00:00
3,35406572,676145,M,1988.0,2024-09-01 00:03:00,01/09/2024 0:10,271,164,2024-09-01 00:00:00
4,35406573,2244256,M,1998.0,2024-09-01 00:03:00,01/09/2024 0:19,199,63,2024-09-01 00:00:00
...,...,...,...,...,...,...,...,...,...
8943936,45705286,2211176,M,1995.0,2026-08-31 23:58:29,2026-09-01 00:09:52,408,72,2026-08-31 23:00:00
8943937,45705287,4612440,M,1990.0,2026-08-31 23:58:53,2026-09-01 00:05:20,32,74,2026-08-31 23:00:00
8943938,45705288,4745711,M,1989.0,2026-08-31 23:58:56,2026-08-31 23:59:04,386,386,2026-08-31 23:00:00
8943939,45705291,4745711,M,1989.0,2026-08-31 23:59:27,2026-09-01 00:08:31,386,384,2026-08-31 23:00:00


In [152]:
cols = ['viaje_id', 'hora_del_viaje', 'origen_id']

df_viajes = ( df_p1[cols]
    .groupby(['hora_del_viaje','origen_id'])
    .count()
    .reset_index()
    .rename(columns={
        'viaje_id':'cantidad_de_viajes'
    })
)

df_viajes

,hora_del_viaje,origen_id,cantidad_de_viajes
0,2024-09-01 00:00:00,4,2
1,2024-09-01 00:00:00,6,1
2,2024-09-01 00:00:00,9,1
3,2024-09-01 00:00:00,10,1
4,2024-09-01 00:00:00,11,3
...,...,...,...
3039645,2026-08-31 23:00:00,408,2
3039646,2026-08-31 23:00:00,412,1
3039647,2026-08-31 23:00:00,424,1
3039648,2026-08-31 23:00:00,455,1


In [173]:
start_date = df_viajes.hora_del_viaje.iloc[0]
end_date = df_viajes.hora_del_viaje.iloc[-1]

df_datetime = pandas.date_range(start=start_date, end=end_date, freq='h').to_frame()

In [176]:
df_ids = pandas.DataFrame(df_p1.origen_id.sort_values().unique(), columns=['origen_id'])

In [180]:
df_hora_estacion = df_datetime.join(df_ids, how='cross').rename(columns={0:'hora_del_viaje'})
df_hora_estacion

,hora_del_viaje,origen_id
0,2024-09-01 00:00:00,2
1,2024-09-01 00:00:00,3
2,2024-09-01 00:00:00,4
3,2024-09-01 00:00:00,5
4,2024-09-01 00:00:00,6
...,...,...
7726315,2026-08-31 23:00:00,478
7726316,2026-08-31 23:00:00,479
7726317,2026-08-31 23:00:00,480
7726318,2026-08-31 23:00:00,481


In [187]:
df_p1_feature = pandas.merge(
    left=df_hora_estacion,
    right=df_viajes,
    left_on=['hora_del_viaje', 'origen_id'],
    right_on=['hora_del_viaje', 'origen_id'],
    suffixes=['', '_v'],
    how='left'
)

df_p1_feature['cantidad_de_viajes'] = (
    df_p1_feature['cantidad_de_viajes']
    .fillna(0)
    .astype('int8')
)

df_p1_feature

,hora_del_viaje,origen_id,cantidad_de_viajes
0,2024-09-01 00:00:00,2,0
1,2024-09-01 00:00:00,3,0
2,2024-09-01 00:00:00,4,2
3,2024-09-01 00:00:00,5,0
4,2024-09-01 00:00:00,6,1
...,...,...,...
7726315,2026-08-31 23:00:00,478,0
7726316,2026-08-31 23:00:00,479,0
7726317,2026-08-31 23:00:00,480,0
7726318,2026-08-31 23:00:00,481,0


## P2 - Clasificación

¿Una estación estará en riesgo de quedarse sin bicicletas o sin anclajes disponibles en un periodo futuro?

Variable objetivo: Riesgo (categórica)
Clases: riesgo de vaciarse, riesgo de llenarse, sin riesgo.

Para construir la variable objetivo, el día se divide en 8 intervalos de 3 horas y cada viaje se asigna dos veces:
- Como salida de su estación de origen
- Como llegada de su estación de destino

Se suman las salidas y las llegadas de cada estación en cada intervalo, y se calcula el flujo neto por estación e intervalo utilizando: llegadas menos salidas.

Las combinaciones de estación e intervalo sin viajes se registran como cero. Por último, el flujo neto se compara con el 40 % de la capacidad de la estación: 
- Riesgo de vaciarse: flujo neto menor o igual a -40% de la capacidad.
- Riesgo de llenarse: flujo neto mayor o igual a +40% de la capacidad.
- Sin riesgo: el flujo neto está entre -40% y +40% de la capacidad, sin llegar a ninguno de los dos.

In [188]:
df_p2 = df_filtered.copy()

### Convert str to datetime

In [192]:
def str_to_datetime(df, col):
    df_str = df[[col]].copy()

    dates_f1_series = pandas.to_datetime(
        df_str.query(f'{col}.str.len() < 19')[col],
        format='%d/%m/%Y %H:%M'
    )
    dates_f2_series = pandas.to_datetime(
        df_str.query(f'{col}.str.len() == 19')[col],
        format='%Y-%m-%d %H:%M:%S'
    )

    dates_f1_series.name = 'date_f1'
    dates_f2_series.name = 'date_f2'

    df_str = df_str.join([dates_f1_series, dates_f2_series])

    datetime_series = df_str.date_f1.fillna( df_str.date_f2 )

    return datetime_series

In [193]:
cols = [
    'inicio_del_viaje',
    'fin_del_viaje'
]

for col in cols:
    df_p2[col] = str_to_datetime(df_p2, col)

In [194]:
df_p2.info()

<class 'pandas.DataFrame'>
Index: 8926227 entries, 0 to 8943940
Data columns (total 8 columns):
 #   Column             Dtype         
---  ------             -----         
 0   viaje_id           int64         
 1   usuario_id         int64         
 2   genero             str           
 3   ano_de_nacimiento  float16       
 4   inicio_del_viaje   datetime64[us]
 5   fin_del_viaje      datetime64[us]
 6   origen_id          int64         
 7   destino_id         int64         
dtypes: datetime64[us](2), float16(1), int64(4), str(1)
memory usage: 571.3 MB


### Group by interval

In [196]:
cols = [
    'inicio_del_viaje',
    'fin_del_viaje'
]

for col in cols:
    df_p2[col] = df_p2[col].dt.floor('3h')

df_p2.inicio_del_viaje.unique()

<DatetimeArray>
['2024-09-01 00:00:00', '2024-09-01 03:00:00', '2024-09-01 06:00:00',
 '2024-09-01 09:00:00', '2024-09-01 12:00:00', '2024-09-01 15:00:00',
 '2024-09-01 18:00:00', '2024-09-01 21:00:00', '2024-09-02 00:00:00',
 '2024-09-02 03:00:00',
 ...
 '2026-08-30 18:00:00', '2026-08-30 21:00:00', '2026-08-31 00:00:00',
 '2026-08-31 03:00:00', '2026-08-31 06:00:00', '2026-08-31 09:00:00',
 '2026-08-31 12:00:00', '2026-08-31 15:00:00', '2026-08-31 18:00:00',
 '2026-08-31 21:00:00']
Length: 5837, dtype: datetime64[us]

In [203]:
cols = ['viaje_id', 'fin_del_viaje', 'origen_id']

df_llegadas = ( df_p2[cols]
    .groupby(['fin_del_viaje','origen_id'])
    .count()
    .reset_index()
    .rename(columns={
        'viaje_id':'llegadas',
        'fin_del_viaje':'hora_del_viaje'
    })
)

df_llegadas

,hora_del_viaje,origen_id,llegadas
0,2024-09-01 00:00:00,4,2
1,2024-09-01 00:00:00,6,1
2,2024-09-01 00:00:00,9,1
3,2024-09-01 00:00:00,10,1
4,2024-09-01 00:00:00,11,3
...,...,...,...
1477319,2026-09-01 00:00:00,350,1
1477320,2026-09-01 00:00:00,386,1
1477321,2026-09-01 00:00:00,408,2
1477322,2026-09-01 03:00:00,331,1


In [204]:
cols = ['viaje_id', 'inicio_del_viaje', 'origen_id']

df_salidas = ( df_p2[cols]
    .groupby(['inicio_del_viaje','origen_id'])
    .count()
    .reset_index()
    .rename(columns={
        'viaje_id':'salidas',
        'inicio_del_viaje':'hora_del_viaje'
    })
)

df_salidas

,hora_del_viaje,origen_id,salidas
0,2024-09-01 00:00:00,4,2
1,2024-09-01 00:00:00,6,1
2,2024-09-01 00:00:00,9,1
3,2024-09-01 00:00:00,10,1
4,2024-09-01 00:00:00,11,3
...,...,...,...
1475195,2026-08-31 21:00:00,456,1
1475196,2026-08-31 21:00:00,462,1
1475197,2026-08-31 21:00:00,464,1
1475198,2026-08-31 21:00:00,480,1


### Fill nans with zeros

In [200]:
start_date = df_p2.inicio_del_viaje.iloc[0]
end_date = df_p2.inicio_del_viaje.iloc[-1]

df_datetime = pandas.date_range(start=start_date, end=end_date, freq='3h').to_frame()

In [201]:
df_ids = pandas.DataFrame(df_p2.origen_id.sort_values().unique(), columns=['origen_id'])

In [202]:
df_hora_estacion = df_datetime.join(df_ids, how='cross').rename(columns={0:'hora_del_viaje'})

In [205]:
df_p2_feature = pandas.merge(
    left=df_hora_estacion,
    right=df_llegadas,
    left_on=['hora_del_viaje', 'origen_id'],
    right_on=['hora_del_viaje', 'origen_id'],
    suffixes=['', '_v'],
    how='left'
)

df_p2_feature = pandas.merge(
    left=df_p2_feature,
    right=df_salidas,
    left_on=['hora_del_viaje', 'origen_id'],
    right_on=['hora_del_viaje', 'origen_id'],
    suffixes=['', '_v'],
    how='left'
)

df_p2_feature

,hora_del_viaje,origen_id,llegadas,salidas
0,2024-09-01 00:00:00,2,NaN,NaN
1,2024-09-01 00:00:00,3,NaN,NaN
2,2024-09-01 00:00:00,4,2.0,2.0
3,2024-09-01 00:00:00,5,NaN,NaN
4,2024-09-01 00:00:00,6,1.0,1.0
...,...,...,...,...
2575435,2026-08-31 21:00:00,478,NaN,NaN
2575436,2026-08-31 21:00:00,479,NaN,NaN
2575437,2026-08-31 21:00:00,480,1.0,1.0
2575438,2026-08-31 21:00:00,481,1.0,1.0


In [206]:
cols = ['llegadas', 'salidas']

for col in cols:
    df_p2_feature[col] = (
        df_p2_feature[col]
        .fillna(0)
        .astype('int8')
    )

df_p2_feature

,hora_del_viaje,origen_id,llegadas,salidas
0,2024-09-01 00:00:00,2,0,0
1,2024-09-01 00:00:00,3,0,0
2,2024-09-01 00:00:00,4,2,2
3,2024-09-01 00:00:00,5,0,0
4,2024-09-01 00:00:00,6,1,1
...,...,...,...,...
2575435,2026-08-31 21:00:00,478,0,0
2575436,2026-08-31 21:00:00,479,0,0
2575437,2026-08-31 21:00:00,480,1,1
2575438,2026-08-31 21:00:00,481,1,1


### Calculate variables

In [232]:
df_p2_feature['flujo'] = df_p2_feature['llegadas'] - df_p2_feature['salidas']

cols = ['dpcapacity']
df_p2_feature = df_p2_feature.join(
    df_nom.set_index('id')[cols],
    on='origen_id',
    how='left',
    rsuffix='_nom'
)

df_p2_feature.head()

,hora_del_viaje,origen_id,llegadas,salidas,flujo,dpcapacity
0,2024-09-01,2,0,0,0,15.0
1,2024-09-01,3,0,0,0,15.0
2,2024-09-01,4,2,2,0,19.0
3,2024-09-01,5,0,0,0,19.0
4,2024-09-01,6,1,1,0,11.0


In [236]:
df_p2_feature['riesgo'] = df_p2_feature.flujo / (df_p2_feature.dpcapacity * 0.4)
df_p2_feature['riesgo'] = np.sign(df_p2_feature.riesgo)
df_p2_feature['riesgo'] = df_p2_feature['riesgo'].fillna(0).astype('int8')

df_p2_feature.head()

,hora_del_viaje,origen_id,llegadas,salidas,flujo,dpcapacity,riesgo
0,2024-09-01,2,0,0,0,15.0,0
1,2024-09-01,3,0,0,0,15.0,0
2,2024-09-01,4,2,2,0,19.0,0
3,2024-09-01,5,0,0,0,19.0,0
4,2024-09-01,6,1,1,0,11.0,0


In [239]:
df_p2_feature['riesgo'] = df_p2_feature.riesgo.replace({
    -1:'riesgo de vaciarse',
    0:'sin riesgo',
    1:'riesgo de llenarse'
})

In [240]:
df_p2_feature.riesgo.value_counts()

riesgo
sin riesgo            1994093
riesgo de llenarse     291113
riesgo de vaciarse     290234
Name: count, dtype: int64

## P3 - Clustering

¿Qué grupos de estaciones comparten patrones de uso y dónde se ubican en la ciudad?

No posee una variable objetivo. Lo que se busca son los patrones ocultos de uso similar entre estaciones que mostrará el clustering.

El algoritmo agrupa estaciones con perfiles parecidos basados en los movimientos de la estación. Un movimiento es cada vez que una bicicleta sale o llega a una estación. Cada viaje posee dos movimientos, con estos se calculan las variables que serán usadas en el clustering:
1. Porcentaje de movimientos, por estación y período del día (mañana, mediodía, tarde y noche)
2. Porcentaje de movimientos en días laborales, por ejemplo, 3 movimientos el lunes y 6 el sábado, sería 3/(3+6); lo que corresponde a 33.33% de movimientos en días laborales.
3. Uso diario por anclaje: es el número promedio de movimientos de una estación por día, dividido entre su capacidad (número de anclajes).
4. Balance de la mañana: indica si la estación gana o pierde bicicletas entre las 5:00 y las 12:00. Se calcula como (llegadas − salidas) ÷ (llegadas + salidas). Un valor negativo significa que salen más bicicletas de las que llegan en la mañana.
5. Balance de la tarde: el mismo cálculo anterior, con los movimientos de 16:00 a 20:00.


In [241]:
df_p3 = df_filtered.copy()

In [242]:
cols = [
    'inicio_del_viaje',
    'fin_del_viaje'
]

for col in cols:
    df_p3[col] = str_to_datetime(df_p3, col)

In [243]:
df_p3.info()

<class 'pandas.DataFrame'>
Index: 8926227 entries, 0 to 8943940
Data columns (total 8 columns):
 #   Column             Dtype         
---  ------             -----         
 0   viaje_id           int64         
 1   usuario_id         int64         
 2   genero             str           
 3   ano_de_nacimiento  float16       
 4   inicio_del_viaje   datetime64[us]
 5   fin_del_viaje      datetime64[us]
 6   origen_id          int64         
 7   destino_id         int64         
dtypes: datetime64[us](2), float16(1), int64(4), str(1)
memory usage: 571.3 MB


In [245]:
df_p3.head()

c:\Users\gsaen\Documents\Maestria ITESM\4. Trimestre 4 (Sep-Nov 2026)\Materias\MLOps\Proyecto\mlops\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,viaje_id,usuario_id,genero,ano_de_nacimiento,inicio_del_viaje,fin_del_viaje,origen_id,destino_id
0,35406564,3366663,M,2000.0,2024-09-01 00:00:00,2024-09-01 00:07:00,154,161
1,35406567,631455,M,1990.0,2024-09-01 00:02:00,2024-09-01 00:08:00,50,156
2,35406571,216004,M,1990.0,2024-09-01 00:02:00,2024-09-01 00:18:00,64,273
3,35406572,676145,M,1988.0,2024-09-01 00:03:00,2024-09-01 00:10:00,271,164
4,35406573,2244256,M,1998.0,2024-09-01 00:03:00,2024-09-01 00:19:00,199,63


### 1 - Porcentaje de movimientos, por estacion y periodo del dia

In [244]:
DAY_PERIODS = {
    "morning": (5, 12),
    "midday": (12, 16),
    "afternoon": (16, 20),
    "night": (20, 5)
}

In [275]:
cols = ['origen_id', 'destino_id']
df_ids = pandas.concat(
    [ df_p3[[col]].rename(columns={col:'id'}) for col in cols ],
    axis=0,
    ignore_index=True
)

df_p3_feature = pandas.DataFrame(df_ids.id.sort_values().unique(), columns=['id'])

df_p3_feature

,id
0,2
1,3
2,4
3,5
4,6
...,...
436,478
437,479
438,480
439,481


In [276]:
col_groups = {
    'inicio': ('inicio_del_viaje', 'origen_id'),
    'fin': ('fin_del_viaje', 'destino_id'),
}

for period, hours in DAY_PERIODS.items():
    df_period = df_p3_feature.copy()
    for k, cols in col_groups.items():
        col_dt = cols[0]
        col_id = cols[1]
        start_hour = hours[0]
        end_hour = hours[1]
        df_range = df_p3.query(f'@start_hour <= {col_dt}.dt.hour < @end_hour')
        df_group = (
            df_range[['viaje_id', col_id]]
            .groupby(col_id)
            .count()
            .rename(columns={'viaje_id':k})
        )
        df_period = df_period.join(df_group, on='id', how='left')
    df_period[period] = (df_period['inicio'] + df_period['fin']).fillna(0).astype('int64')
    df_p3_feature = df_p3_feature.join(df_period[[period]], how='left')

df_p3_feature

,id,morning,midday,afternoon,night
0,2,28192,17126,22881,0
1,3,22719,15218,19651,0
2,4,29245,18666,26365,0
3,5,28052,18061,17605,0
4,6,14336,10575,12616,0
...,...,...,...,...,...
436,478,225,156,242,0
437,479,190,136,189,0
438,480,158,117,152,0
439,481,171,127,148,0


In [277]:
df_p3_feature['total'] = df_p3_feature.morning + df_p3_feature.midday + df_p3_feature.afternoon + df_p3_feature.night

for period in DAY_PERIODS:
    period_col = period + '_pct'
    df_p3_feature[period_col] = df_p3_feature[period] / df_p3_feature['total'] * 100

df_p3_feature.drop(columns=[ 'total', *(DAY_PERIODS.keys()) ], inplace=True)

df_p3_feature

,id,morning_pct,midday_pct,afternoon_pct,night_pct
0,2,41.337850,25.111805,33.550345,0.0
1,3,39.450927,26.425644,34.123428,0.0
2,4,39.373418,25.130594,35.495988,0.0
3,5,44.025236,28.345209,27.629555,0.0
4,6,38.201828,28.179711,33.618461,0.0
...,...,...,...,...,...
436,478,36.115570,25.040128,38.844302,0.0
437,479,36.893204,26.407767,36.699029,0.0
438,480,37.002342,27.400468,35.597190,0.0
439,481,38.340807,28.475336,33.183857,0.0


### 2 - Porcentaje de movimientos en dias laborales

In [ ]:
df_p3

c:\Users\gsaen\Documents\Maestria ITESM\4. Trimestre 4 (Sep-Nov 2026)\Materias\MLOps\Proyecto\mlops\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,viaje_id,usuario_id,genero,ano_de_nacimiento,inicio_del_viaje,fin_del_viaje,origen_id,destino_id
0,35406564,3366663,M,2000.0,2024-09-01 00:00:00,2024-09-01 00:07:00,154,161
1,35406567,631455,M,1990.0,2024-09-01 00:02:00,2024-09-01 00:08:00,50,156
2,35406571,216004,M,1990.0,2024-09-01 00:02:00,2024-09-01 00:18:00,64,273
3,35406572,676145,M,1988.0,2024-09-01 00:03:00,2024-09-01 00:10:00,271,164
4,35406573,2244256,M,1998.0,2024-09-01 00:03:00,2024-09-01 00:19:00,199,63
...,...,...,...,...,...,...,...,...
8943936,45705286,2211176,M,1995.0,2026-08-31 23:58:29,2026-09-01 00:09:52,408,72
8943937,45705287,4612440,M,1990.0,2026-08-31 23:58:53,2026-09-01 00:05:20,32,74
8943938,45705288,4745711,M,1989.0,2026-08-31 23:58:56,2026-08-31 23:59:04,386,386
8943939,45705291,4745711,M,1989.0,2026-08-31 23:59:27,2026-09-01 00:08:31,386,384


### 3 - Uso diario por anclaje

### 4 - Balance de la mañana

### 5 - Balance de la tarde